[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Selecting Rows


## What you will be able to do

Build a `where` clause that means what it looks like, and say why `and` and a misplaced `&` both
return the wrong rows without raising. Read the SQL of any query before it runs. Count rows, ask
whether any match, and total a column, all in the database rather than in Python. Group rows and
filter the groups. Ask for one row and choose what happens when there is none. Draw pages that do
not show the same row twice while the table is being written to. Drop to SQL when the builder runs
out, and say what `raw` gives back that `execute_sql` does not.


## The idea

### The problem

`where` takes an expression, and the expression is built with Python operators. Two of the operators
you would reach for first are the wrong ones, and neither mistake raises.

`and` cannot be overloaded. Python decides what `a and b` means by asking `a` whether it is truthy,
and a peewee expression is always truthy, so `a and b` evaluates to `b` and the first half of your
filter is discarded before peewee ever sees it. The query runs. It returns rows. They are the wrong
rows.

`&` can be overloaded, and peewee overloads it, but `&` binds more tightly than `>` and `==`. So
`Book.year >= 2020 & Book.pages > 200` groups as `Book.year >= (2020 & Book.pages) > 200`, which is
a different question, and usually one that no row answers. That query runs too.

### What a query is

`Book.select()` returns a query object, not rows. Methods like `where`, `order_by` and `limit`
return a new query, so a query is built up by chaining and sends nothing until you iterate it, count
it or ask for one row from it. That is what makes `sql()` useful: a query can be printed and read
before it is ever run.

### Why it works that way

Python lets a class define `__and__`, `__or__` and `__invert__`, which is why peewee can give `&`,
`|` and `~` a meaning. It does not let a class define `and`, `or` or `not`, because those are
control flow: they decide whether to evaluate the right operand at all. The only hook they offer is
`__bool__`, which answers a question peewee cannot usefully answer about an unrun query. So the
operators that read best are exactly the ones that cannot work, and the ones that work carry
arithmetic precedence that has nothing to do with logic. Parentheses are how you take that
precedence back.

### Where this shows up

Every filter built from more than one condition, which is most filters past the first week. It
shows up worst in code that was tested against data where the two conditions happened to agree,
because then the wrong spelling returns the right rows and nothing tells you.

### What this notebook covers

Reading a query as SQL before running it. The three operators and the parentheses they need.
Ordering, and the shape `order_by` will not take. Counting, existence and totals computed by the
database. Grouping with `fn.COUNT`, and filtering groups with `having`. Asking for one row three
ways. Pages, and the way offset paging repeats a row when the table changes underneath it. Raw SQL,
in the two forms peewee offers. Then the four failures, two of them silent.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Book(Model):
    title = CharField()
    year = IntegerField()
    pages = IntegerField()

    class Meta:
        database = db


db.create_tables([Book])
for title, year, pages in [("Nightjar", 2018, 244), ("Harmattan", 2019, 331),
                           ("The Quiet Engine", 2021, 398), ("Riverwork", 2022, 189)]:
    Book.create(title=title, year=year, pages=pages)

# wanted: the long books from 2020 on, which is The Quiet Engine and nothing else
spellings = [("and", Book.year >= 2020 and Book.pages > 200),
             ("&", Book.year >= 2020 & Book.pages > 200),
             ("(...) & (...)", (Book.year >= 2020) & (Book.pages > 200))]

for label, expression in spellings:
    query = Book.select().where(expression)
    print(f"{label:<14} -> {[book.title for book in query]}")
    print(f"{'':<17} WHERE {query.sql()[0].split('WHERE ')[1]}")
```

```
and            -> ['Nightjar', 'Harmattan', 'The Quiet Engine']
                  WHERE ("t1"."pages" > ?)
&              -> []
                  WHERE ((? AND "t1"."pages") > ?)
(...) & (...)  -> ['The Quiet Engine']
                  WHERE (("t1"."year" >= ?) AND ("t1"."pages" > ?))
```

Three spellings of one filter, three different answers, and no exceptions. The `and` form lost the
year condition on the way in, so it returned two books from before 2020. The `&` form built a
question about a bitwise `AND` of the number 2020 with a column, which nothing matched. Only the
parenthesized form asked what was meant. The `WHERE` clause is the evidence, and printing it is the
habit this notebook is trying to build.


## Setup

Five imports, peewee installed and pinned, the catalog's models, and the catalog loaded.

- `peewee` is the library, and `Model`, the field classes and `SqliteDatabase`, from it, are what a
  model is written with
- `fn`, also from peewee, is how a SQL function such as `COUNT` or `SUM` is written in a query
- `OperationalError` is caught once, to show a shape `order_by` will not take
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `AUTHORS` and `BOOKS` are the catalog, and `build` makes the tables and loads them
- `sql` prints the SQL a query will send, with the values that go beside it

The catalog is the same four authors and twelve books as the rest of the guide, in a database in
memory. Twelve rows is small enough that every answer below can be checked by eye, which is the
point: a query you can verify by counting on your fingers is the only kind worth learning a new
operator on.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, Model, OperationalError,
                    SqliteDatabase, fn)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

db = SqliteDatabase(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books")


peewee 4.5.1 | the catalog: 4 authors and 12 books


## Worked examples

### A query is a description until you ask it for rows

`select` and the methods that follow it return query objects. Nothing has been sent to the database
in this cell:


In [2]:
recent = Book.select().where(Book.year >= 2015)

print("the type:", type(recent).__name__)
print(sql(recent))
print("still nothing has run")


the type: ModelSelect
SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."year" >= ?)  [2015]
still nothing has run


Each method returned a new query rather than changing the old one, and the SQL is there to read
before a row exists. Iterating is what runs it:


In [3]:
print("titles:", [book.title for book in recent])
print("how many:", recent.count())


titles: ['The Drum Line', 'The Lantern Keeper', 'Nightjar', 'Harmattan', 'The Quiet Engine', 'Riverwork', 'Small Machines']
how many: 7


### The three operators, and the parentheses

`&` is and, `|` is or, `~` is not. Each comparison gets its own parentheses, every time:

| What you mean | How to write it |
|---|---|
| both | `(Book.year >= 2015) & (Book.pages > 300)` |
| either | `(Book.year < 2000) \| (Book.pages > 400)` |
| not | `~(Book.year == 2018)` |
| a set | `Book.year.in_([2018, 2019])` |
| a range | `Book.year.between(2010, 2015)` |
| text | `Book.title.startswith("The")`, `Book.title.contains("or")` |


In [4]:
both = Book.select().where((Book.year >= 2015) & (Book.pages > 300))
either = Book.select().where((Book.year < 2000) | (Book.pages > 400))
neither = Book.select().where(~(Book.title.startswith("The")))

for label, query in (("both", both), ("either", either), ("not", neither)):
    print(f"  {label:<8} {len(query):>2}  {[book.title for book in query][:3]}")
print()
print(sql(either))


  both      2  ['Harmattan', 'The Quiet Engine']
  either    2  ['Stone and Tide', 'A Careful Fire']
  not       7  ['Nightjar', 'Stone and Tide', 'Riverwork']

SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE (("t1"."year" < ?) OR ("t1"."pages" > ?))  [2000, 400]


`len(query)` runs it and counts the rows it got back, which is different from `.count()`, and the
difference matters further down.

### Ordering

`order_by` takes the columns as separate arguments, and `.desc()` reverses one of them:


In [5]:
longest = Book.select().order_by(Book.pages.desc()).limit(3)
print("longest:", [(book.title, book.pages) for book in longest])

by_year = Book.select().order_by(Book.year, Book.id)
print("oldest three:", [(book.title, book.year) for book in by_year][:3])
print(sql(by_year))


longest: [('Stone and Tide', 501), ('A Careful Fire', 420), ('The Quiet Engine', 398)]
oldest three: [('A Careful Fire', 1998), ('The Long Field', 2004), ('Stone and Tide', 2009)]
SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" ORDER BY "t1"."year", "t1"."id"


Passing those columns as a list instead of as separate arguments builds a different thing: SQLite
reads `(year, id)` as a row value, which it will not order by.


In [6]:
try:
    list(Book.select().order_by([Book.year, Book.id]))
except OperationalError as error:
    print("peewee.OperationalError:", error)
print("the SQL it built:", Book.select().order_by([Book.year, Book.id]).sql()[0].split("ORDER BY ")[1])


peewee.OperationalError: row value misused
the SQL it built: ("t1"."year", "t1"."id")


The `ORDER BY ("year", "id")` with one pair of parentheses is the whole difference. `order_by(a, b)`
is the spelling that works.

### Counting, existence and totals, without moving rows

Three questions the database answers with one number each:


In [7]:
print("count()  :", Book.select().where(Book.year >= 2015).count())
print("exists() :", Book.select().where(Book.pages > 500).exists())
print("scalar() :", Book.select(fn.SUM(Book.pages)).scalar(), "pages in all")
print(sql(Book.select(fn.SUM(Book.pages))))


count()  : 7
exists() : True
scalar() : 3774 pages in all
SELECT SUM("t1"."pages") FROM "book" AS "t1"


`count()` wraps the query in a `SELECT COUNT(1)`, so the rows are never sent. `exists()` adds a
`LIMIT 1` and asks only whether anything came back. `scalar()` returns the first column of the first
row, which is what an aggregate with no `group_by` produces.

`len(list(query))` answers the same question as `count()` by fetching every row and building every
object, which is fine for twelve books and wrong for a million.

### Grouping, and filtering the groups

`fn.COUNT` with `group_by` gives one row per group, and `alias` names the column so it can be read
off the result:


In [8]:
per_author = (Author.select(Author.name, fn.COUNT(Book.id).alias("books"),
                            fn.SUM(Book.pages).alias("pages"))
                    .join(Book)
                    .group_by(Author.name)
                    .order_by(Author.name))

for row in per_author:
    print(f"  {row.name:<15} {row.books} books, {row.pages} pages")


  Ines O'Brien    3 books, 1038 pages
  Kofi Mensah     3 books, 816 pages
  Marco Pietra    3 books, 966 pages
  Ursula Vance    3 books, 954 pages


`having` filters the groups, which is a different job from `where`. `where` chooses which rows go
into a group, and `having` chooses which groups survive, so a condition on a count can only go in
`having`:


In [9]:
prolific = (Author.select(Author.name, fn.COUNT(Book.id).alias("books"))
                  .join(Book)
                  .where(Book.year >= 2015)                         # which rows count
                  .group_by(Author.name)
                  .having(fn.COUNT(Book.id) > 1))                   # which groups are kept

print(sql(prolific))
print([(row.name, row.books) for row in prolific])


SELECT "t1"."name", COUNT("t2"."id") AS "books" FROM "author" AS "t1" INNER JOIN "book" AS "t2" ON ("t2"."author_id" = "t1"."id") WHERE ("t2"."year" >= ?) GROUP BY "t1"."name" HAVING (COUNT("t2"."id") > ?)  [2015, 1]
[('Kofi Mensah', 3), ('Marco Pietra', 2), ('Ursula Vance', 2)]


### One row, or none

Three ways to ask for a single row, differing only in what they do when there is not one:


In [10]:
print("get()        :", Book.get(Book.title == "Nightjar").year)
print("get_or_none():", Book.get_or_none(Book.title == "No Such Book"))
print("first()      :", Book.select().where(Book.year > 2100).first())


get()        : 2018
get_or_none(): None
first()      : None


`get` raises when nothing matches, which is right when the row must be there and you want the
program to stop. `get_or_none` returns `None`, which is right when its absence is an ordinary case
you are about to handle. `first` is the same idea on a query you have already built, and it adds the
`LIMIT 1` for you. The exception `get` raises is in the Common errors below.

### Pages

`paginate(page, per_page)` is `limit` and `offset` with the arithmetic done for you:


In [11]:
page = Book.select().order_by(Book.year, Book.id).paginate(2, 4)
print(sql(page))
print("page 2:", [book.title for book in page])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" ORDER BY "t1"."year", "t1"."id" LIMIT ? OFFSET ?  [4, 4]
page 2: ['The Salt Road', 'The Drum Line', 'The Lantern Keeper', 'Nightjar']


Pages are counted from one, and the `OFFSET` is `(page - 1) * per_page`.

Offset paging asks the database to skip a number of rows, and that number is only correct while the
rows before it stay put. Here are two pages drawn a moment apart, with one book written in between:


In [12]:
def titles(number, per_page=4):
    return [book.title for book in Book.select().order_by(Book.year, Book.id).paginate(number, per_page)]


first_page = titles(1)
Book.create(title="An Early Draft", author=Author.get(Author.name == "Ines O'Brien"),
            year=1990, pages=120)                                   # sorts in front of everything
second_page = titles(2)

print("page 1:", first_page)
print("page 2:", second_page)
print("shown twice:", sorted(set(first_page) & set(second_page)))


page 1: ['A Careful Fire', 'The Long Field', 'Stone and Tide', 'Winter Harbour']
page 2: ['Winter Harbour', 'The Salt Road', 'The Drum Line', 'The Lantern Keeper']
shown twice: ['Winter Harbour']


The new book sorted to the front, every row moved one place later, and the row that had been last on
page 1 was still sitting at offset four when page 2 was asked for. The reader sees it twice, and
whichever row was pushed past the end is never seen at all. Nothing here is a peewee problem: it is
what `OFFSET` means.

Paging by the last key you saw asks a question whose answer does not move:


In [13]:
page_one = list(Book.select().order_by(Book.id).limit(4))
Book.create(title="Another Draft", author=Author.get(Author.name == "Ines O'Brien"),
            year=1991, pages=130)
page_two = list(Book.select().where(Book.id > page_one[-1].id).order_by(Book.id).limit(4))

print("page 1:", [book.title for book in page_one])
print("page 2:", [book.title for book in page_two])
print("shown twice:", sorted({b.title for b in page_one} & {b.title for b in page_two}))


page 1: ['The Salt Road', 'Nightjar', 'The Quiet Engine', 'Stone and Tide']
page 2: ['The Lantern Keeper', 'Riverwork', 'A Careful Fire', 'The Long Field']
shown twice: []


The cost is that pages have to be walked in order, since there is no way to jump to page nine
without having seen page eight. That is the trade: offset can jump and can repeat itself, and keyset
cannot jump and cannot repeat itself.

### Down to SQL

Two escape hatches, and they give back different things. `raw` returns model instances:


In [14]:
rows = Book.raw("SELECT * FROM book WHERE pages > ? ORDER BY pages", 380)
first = list(rows)[0]
print("type:", type(first).__name__, "| title:", first.title, "| year:", first.year)

cursor = db.execute_sql("SELECT title, pages FROM book WHERE pages > ? ORDER BY pages", (380,))
print("type:", type(cursor).__name__, "| rows:", cursor.fetchall())


type: Book | title: The Quiet Engine | year: 2021
type: Cursor | rows: [('The Quiet Engine', 398), ('A Careful Fire', 420), ('Stone and Tide', 501)]


`raw` builds `Book` objects, so everything a model gives you still works. `execute_sql` hands back
the driver's own cursor and tuples, with no model involved, which is what you want for a statement
that does not correspond to a table at all. Both take the values separately from the SQL, which is
the placeholder the **Why Peewee** notebook started with.

### When to reach for which

| The question | The call |
|---|---|
| how many rows match | `query.count()` |
| is there at least one | `query.exists()` |
| one number from an aggregate | `query.scalar()` |
| one row that must be there | `Model.get(...)` |
| one row that may not be | `Model.get_or_none(...)`, or `query.first()` |
| one group per value, with totals | `.select(col, fn.COUNT(...).alias(...)).group_by(col)` |
| keep only some groups | `.having(...)` |
| a page a reader can jump to | `.paginate(page, per_page)` |
| a page that cannot repeat a row | `.where(Model.id > last_seen).limit(n)` |
| SQL the builder will not write, as objects | `Model.raw(...)` |
| SQL with no model behind it | `db.execute_sql(...)` |

`count` and `get_or_none` are the defaults: reach for `len(list(query))` and `get` only when you
have a reason. For pages, `paginate` is right for a catalog a person clicks through and keyset is
right for a job that walks the whole table.

### A catalog page, finished

One function, holding everything above: a filter built with parentheses, an order that breaks ties,
a total counted in the database, and a page that cannot show a row twice.


In [15]:
def catalog_page(after=None, per_page=4, since=2000, longer_than=150):
    """One page of the catalog, oldest first, with the total that matched."""
    matching = (Book.year >= since) & (Book.pages > longer_than)     # parentheses, every time
    found = Book.select().where(matching)

    page = found.where(Book.id > after) if after is not None else found
    rows = list(page.order_by(Book.id).limit(per_page))
    return rows, found.count()


seen, total = None, None
for number in (1, 2, 3):
    rows, total = catalog_page(after=seen)
    seen = rows[-1].id if rows else seen
    print(f"  page {number}: {[book.title for book in rows]}")
print("  matching in all:", total)


  page 1: ['The Salt Road', 'Nightjar', 'The Quiet Engine', 'Stone and Tide']
  page 2: ['The Lantern Keeper', 'Riverwork', 'The Long Field', 'Winter Harbour']
  page 3: ['The Drum Line', 'Harmattan', 'Small Machines']
  matching in all: 11


The `found.count()` is computed from the filtered query without the paging, so the total describes
the whole result rather than the page. The page itself is bounded by the last `id` the caller saw,
so a book written while a reader is on page 2 changes what page 3 holds without ever pushing a row
across a boundary.

### Where each part came from

| In the catalog page | What it relies on | The section that showed it |
|---|---|---|
| `(Book.year >= since) & (Book.pages > longer_than)` | parentheses around every comparison | The three operators |
| `Book.select().where(matching)` | a query that has not run yet | A query is a description |
| `found.where(Book.id > after)` | a query built on another query | A query is a description |
| `.order_by(Book.id).limit(per_page)` | an order with no ties, and a bound | Ordering, and Pages |
| `found.count()` | a number the database computes | Counting, existence and totals |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/04-selecting-rows-solutions.ipynb).

**1.** Find the books from 2015 on that are longer than 250 pages, print the SQL first, and then the
titles.


In [16]:
# your code here


**2.** Write the same filter with Python's `and` instead, print its SQL, and say which condition was
lost.


In [17]:
# your code here


**3.** Print each author's name with the year of their earliest book, using `fn.MIN` and `group_by`,
ordered by that year.


In [18]:
# your code here


**4.** Print the authors with more than two books published from 2010 on, using `where` to
choose the rows and `having` to choose the groups.


In [19]:
# your code here


**5.** Ask for a book that does not exist in the two ways that do not raise, and print what each one
gives back.


In [20]:
# your code here


**6.** Walk the whole catalog in pages of five by the last `id` seen, printing each page, and show
that no title appears on two pages.


In [21]:
# your code here


## Common errors

### No error, and rows from before the year you asked for: Python's and in a where


In [22]:
wrong = Book.select().where(Book.year >= 2020 and Book.pages > 200)

print(sql(wrong))
print("rows:", len(wrong), "| of them from before 2020:",
      len([book for book in wrong if book.year < 2020]))
print("oldest returned:", min((book.year, book.title) for book in wrong))


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."pages" > ?)  [200]
rows: 10 | of them from before 2020: 8
oldest returned: (1998, 'A Careful Fire')


The `WHERE` clause has one condition in it, and the query was written with two. `and` asked the left
expression whether it was truthy, got `True`, and evaluated to the right expression alone. peewee was
handed a filter that had already lost half of itself.

Nothing can warn you. By the time `where` is called there is only one expression, and it is a
perfectly good one. The fix is the operator that can be overloaded, with parentheses:


In [23]:
right = Book.select().where((Book.year >= 2020) & (Book.pages > 200))

print(sql(right))
print("rows:", len(right), "|", [(book.title, book.year) for book in right])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE (("t1"."year" >= ?) AND ("t1"."pages" > ?))  [2020, 200]
rows: 2 | [('The Quiet Engine', 2021), ('Small Machines', 2023)]


### No error, and no rows at all: & binding tighter than the comparison


In [24]:
empty = Book.select().where(Book.year >= 2020 & Book.pages > 200)

print(sql(empty))
print("rows:", len(empty))


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ((? AND "t1"."pages") > ?)  [2020, 200]
rows: 0


`&` is an arithmetic operator in Python, and arithmetic binds tighter than comparison. So the
grouping is `Book.year >= (2020 & Book.pages) > 200`, and what reaches the database is a comparison
against a bitwise `AND` of a number with a column.

An empty result is the usual symptom, which is worse than it sounds: an empty list is what a filter
that simply matched nothing looks like, so this one gets explained away rather than found. Print the
`WHERE` clause whenever a filter returns nothing and you expected rows.

### BookDoesNotExist: <Model: Book> instance matching query does not exist:


In [25]:
Book.get(Book.title == "The Book That Was Never Written")


BookDoesNotExist: <Model: Book> instance matching query does not exist:
SQL: SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."title" = ?) LIMIT ? OFFSET ?
Params: ['The Book That Was Never Written', 1, 0]

Every model gets its own exception class, named after it, so the traceback says which table was
asked. peewee puts the SQL and the values into the message, which means the failing query is in
front of you without any extra work.

`get` is the right call when the row must exist. When it may not, say so:


In [26]:
missing = Book.get_or_none(Book.title == "The Book That Was Never Written")
print("get_or_none:", missing)
print("first():    ", Book.select().where(Book.title == "The Book That Was Never Written").first())
print("a real one: ", Book.get_or_none(Book.title == "Nightjar").year)


get_or_none: None
first():     None
a real one:  2018


### peewee.ProgrammingError: Incorrect number of bindings supplied.


In [27]:
db.execute_sql("SELECT title FROM book WHERE pages > ?", (200, 300))


ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 2 supplied.

One placeholder, two values. The message counts both for you. This is the cost of the escape hatch:
inside `execute_sql` the SQL is a string you wrote, and nothing checks it against the values until
the driver does.

The values go in a sequence, and a single value still needs to be a sequence, which is where the
stray comma in `(200,)` earns its place:


In [28]:
print(db.execute_sql("SELECT title FROM book WHERE pages > ?", (380,)).fetchall())
print(list(Book.raw("SELECT * FROM book WHERE pages > ?", 380))[0].title)


[('The Quiet Engine',), ('Stone and Tide',), ('A Careful Fire',)]
The Quiet Engine


## Recap

- A query is a description. `select`, `where` and the rest return new queries, and nothing is sent
  until you iterate, count, or ask for one row.
- `sql(query)` prints what will be sent, which is how every claim in this notebook was checked.
- `&`, `|` and `~` are and, or and not, and every comparison needs its own parentheses because those
  operators bind tighter than the comparisons inside them.
- Python's `and` cannot be overloaded, so it silently discards the left half of a filter. Both wrong
  spellings return rows rather than raising.
- `count`, `exists` and `scalar` are answered by the database. `len(list(query))` fetches every row
  to answer the same question.
- `group_by` makes one row per value, `alias` names the computed column, and `having` filters groups
  where `where` filters rows.
- `get` raises, `get_or_none` and `first` return `None`.
- `paginate` can show a row twice when the table changes under it. Paging by the last key seen
  cannot, and cannot jump to a page either.
- `raw` returns model instances and `execute_sql` returns the driver's cursor.


## What is next

The **Transactions** notebook is about writes that have to happen together: `db.atomic` as a block
and as a decorator, savepoints nested inside it, what a rollback does to the instances you are still
holding, and the exception that was caught so carefully that the transaction committed anyway.


---

&#8592; **Previous:** [Creating and Changing Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/03-creating-and-changing-rows.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/05-transactions.ipynb) &#8594;
